<a href="https://colab.research.google.com/github/Takumi173/Test/blob/main/Dataset_JSON_Reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備

## 処理用にデータを結合

In [1]:
# データのコピー
!git clone https://github.com/cdisc-org/sdtm-adam-pilot-project.git

Cloning into 'sdtm-adam-pilot-project'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 224 (delta 64), reused 220 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 24.51 MiB | 3.70 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (87/87), done.


In [2]:
# 使用するjsonデータとdefine.xmlを新規フォルダにコピーする

import os
import shutil
import json

source_dir = "sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm"
json_dir   = "json_files"
define_dir = "define_xml"

if not os.path.exists(json_dir):
    os.makedirs(json_dir)

if not os.path.exists(define_dir):
    os.makedirs(define_dir)

for root, _, files in os.walk(source_dir):
  for file in files:
    if file.endswith(".json"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(json_dir, file)
      shutil.copy(source_path, target_path)
    if file.endswith("define.xml"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(define_dir, file)
      shutil.copy(source_path, target_path)

In [3]:
# jsonファイルをリスト形式に結合したファイル（dataset_list.json）を作成

dataset_list = []
for filename in os.listdir(json_dir):
  if filename.endswith(".json"):
    with open(os.path.join(json_dir, filename), "r") as f:
      try:
        json_data = json.load(f)
        dataset_list.append(json_data)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON in file {filename}: {e}")

with open("dataset_list.json", "w") as f:
  json.dump(dataset_list, f)


## 症例フィルタリング関数の定義

In [4]:
def filter_data(data, target_usubjids):
    """
    複数のドメインデータを含むリストから、指定されたUSUBJIDのrowsのみを抽出して新しいJSONファイルに保存する。
    入力データがリストでない場合はエラーメッセージを出力する。
    データ構造は、"columns" 内の "name" が "USUBJID" の列を持つことを前提とする。

    Args:
        data (list): ドメインを結合させたのリスト。リストでない場合はエラーとなる。
        output_file (str): 出力するJSONファイル名。
        target_usubjids (list): 残したいUSUBJIDのリスト。
    """
    if not isinstance(data, list):
        print("エラー：入力データはJSONオブジェクトのリストである必要があります。")
        return

    filtered_data_list = []
    for item in data:
        usubjid_index = -1
        if 'columns' in item:
            for i, col in enumerate(item['columns']):
                if 'name' in col and col['name'] == 'USUBJID':
                    usubjid_index = i
                    break

        if usubjid_index == -1:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'name' が 'USUBJID' の列が見つかりません。スキップします。")
            filtered_data_list.append(item)
            continue

        if 'rows' in item:
            filtered_rows = [
                row for row in item['rows'] if len(row) > usubjid_index and row[usubjid_index] in target_usubjids
            ]
            new_data = item.copy()
            new_data['rows'] = filtered_rows
            new_data['records'] = len(filtered_rows)
            filtered_data_list.append(new_data)
        else:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'rows' が見つかりません。スキップします。")
            filtered_data_list.append(item)

    return filtered_data_list



# 実行テスト
# with open('dataset_list.json', 'r') as f:
#   data = json.load(f)
#
# target_ids = ['01-701-1211']
# output_filename = 'filtered_list.json'
#
# filtered_data_list = filter_data(data, target_ids)
#
# with open(output_filename, 'w') as f:
#   json.dump(filtered_data_list, f)
#
# print(f"処理完了：'{output_filename}' に USUBJID が {target_ids} のデータを出力しました。")

## データ書き換え関数の定義

In [5]:
def data_update(data, target_domain, target_usubjid, target_seq, target_variable, new_value):
    """
    指定されたUSUBJIDを持つレコードの指定された変数を書き換えます。
    {target_domain}SEQが存在する場合はそれもキーとして使用します。
    元のデータは変更せず、新しいデータ構造を返します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 書き換えたいレコードのUSUBJID。
        target_seq (int): 書き換えたいレコードの{target_domain}SEQの値（存在しない場合は無視されます）。
        target_variable (str): 書き換えたい変数の名前（例: "CMTRT"）。
        new_value (any): 新しい変数の値。

    Returns:
        list: 指定された変数が更新された新しいデータ全体のリスト。
              該当するレコードが見つからなかった場合、元のデータのコピーを返します。
    """
    updated_data = []
    seqname = target_domain + 'SEQ'

    for dataset in data:
        updated_dataset = dataset.copy()
        if updated_dataset.get("itemGroupOID") == target_domain:
            updated_rows = []
            found = False
            usubjid_index = -1
            seq_index = -1
            variable_index = -1
            has_seq = False

            for i, col in enumerate(updated_dataset["columns"]):
                if col["name"] == "USUBJID":
                    usubjid_index = i
                elif col["name"] == seqname:
                    seq_index = i
                    has_seq = True
                elif col["name"] == target_variable:
                    variable_index = i

            if usubjid_index != -1 and variable_index != -1:
                for row in dataset["rows"]:
                    updated_row = list(row)  # 行をコピーして変更
                    usubjid_match = updated_row[usubjid_index] == target_usubjid
                    seq_match = True
                    if has_seq and seq_index != -1:
                        seq_match = (len(updated_row) > seq_index and updated_row[seq_index] == target_seq)
                    elif has_seq:
                        print(f"警告: '{target_domain}' データセットに '{seqname}' 列が見つかりましたが、インデックスが無効です。USUBJIDのみをキーとして使用します。")

                    if usubjid_match and seq_match:
                        updated_row[variable_index] = new_value
                        if has_seq and seq_index != -1:
                            print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' の '{target_variable}' を '{new_value}' に更新しました。")
                        else:
                            print(f"USUBJID '{target_usubjid}' の '{target_variable}' を '{new_value}' に更新しました。")
                        found = True
                    updated_rows.append(updated_row)
                updated_dataset["rows"] = updated_rows
            elif updated_dataset.get("itemGroupOID") == target_domain:
                print(f"'{target_domain}' データセットに 'USUBJID' または '{target_variable}' 列が見つかりませんでした。")

            updated_data.append(updated_dataset)
            if not found and updated_dataset.get("itemGroupOID") == target_domain:
                if has_seq and seq_index != -1:
                    print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' に該当するレコードが見つかりませんでした。")
                else:
                    print(f"USUBJID '{target_usubjid}' に該当するレコードが見つかりませんでした。")
        else:
            updated_data.append(updated_dataset)

    if not any(d.get("itemGroupOID") == target_domain for d in data):
        print(f"{target_domain} データセットが見つかりませんでした。")

    return updated_data

# 書き換えテスト
# updated_data = data_update(filtered_data_list, "DM", "01-701-1211", 0, "AGE", 49)
# updated_data = data_update(updated_data, "CM", "01-701-1211", 3, "CMTRT", "New Drug 123456789")
# updated_data = data_update(updated_data, "CM", "01-701-1211", 0, "CMDOSE", 123)

## データ比較関数の定義

In [6]:
from typing import List, Dict, Any

def compare_data(old_data: List[Dict[str, Any]], new_data: List[Dict[str, Any]]) -> None:
    """
    2つのデータリストの更新差分を人間が読みやすい形式で出力します。

    Args:
        old_data: 旧データリスト。
        new_data: 新データリスト。
    """

    def create_row_dict(item_group: Dict[str, Any], row: List[Any]) -> Dict[str, Any]:
        """rowデータをキー付きの辞書に変換する"""
        row_dict = {}
        for i, column in enumerate(item_group['columns']):
            row_dict[column['name']] = row[i]
        return row_dict

    def get_key_values(item_group_oid: str, row_dict: Dict[str, Any]) -> Dict[str, Any]:
        """データのキーとなる値を抽出する"""
        key_values = {'USUBJID': row_dict.get('USUBJID')}
        seq_key = f"{item_group_oid}SEQ"
        if seq_key in row_dict:
            key_values[seq_key] = row_dict[seq_key]
        return key_values

    def format_key(key_values: Dict[str, Any]) -> str:
        """キー値を人間が読みやすい文字列に整形する"""
        parts = []
        for key, value in key_values.items():
            if value is not None:
                parts.append(f"{key} = {value}")
        return ", ".join(parts)

    old_data_by_group = {item['itemGroupOID']: item for item in old_data}
    new_data_by_group = {item['itemGroupOID']: item for item in new_data}

    all_group_oids = set(old_data_by_group.keys()) | set(new_data_by_group.keys())

    for group_oid in sorted(list(all_group_oids)):
        print(f"--- ItemGroupOID: {group_oid} ---")
        old_group = old_data_by_group.get(group_oid)
        new_group = new_data_by_group.get(group_oid)

        old_rows_by_key = {}
        if old_group and 'rows' in old_group:
            for row in old_group['rows']:
                row_dict = create_row_dict(old_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    old_rows_by_key[format_key(key_values)] = row_dict

        new_rows_by_key = {}
        if new_group and 'rows' in new_group:
            for row in new_group['rows']:
                row_dict = create_row_dict(new_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    new_rows_by_key[format_key(key_values)] = row_dict

        old_keys = set(old_rows_by_key.keys())
        new_keys = set(new_rows_by_key.keys())

        # 追加されたデータ
        added_keys = new_keys - old_keys
        for key in sorted(list(added_keys)):
            print(f"{key}:")
            print("  Added")
            for item_key, old_value in sorted(new_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 削除されたデータ
        removed_keys = old_keys - new_keys
        for key in sorted(list(removed_keys)):
            print(f"{key}:")
            print("  Deleted")
            for item_key, old_value in sorted(old_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 更新されたデータ
        common_keys = old_keys & new_keys
        for key in sorted(list(common_keys)):
            if old_rows_by_key[key] != new_rows_by_key[key]:
                print(f"{key}:")
                print("  Updated:")
                old_row = old_rows_by_key[key]
                new_row = new_rows_by_key[key]
                for item_key in sorted(list(set(old_row.keys()) | set(new_row.keys()))):
                    old_value = old_row.get(item_key)
                    new_value = new_row.get(item_key)
                    if old_value != new_value:
                        print(f"    {item_key}: {old_value!r} -> {new_value!r}")
                print()


# 比較テスト
# compare_data(filtered_data_list, updated_data)

In [7]:
import json

usubjids = set()
for dataset in dataset_list:
    if 'columns' in dataset:
        for i, col in enumerate(dataset['columns']):
            if 'name' in col and col['name'] == 'USUBJID':
                if 'rows' in dataset:
                    for row in dataset['rows']:
                        if len(row) > i:
                            usubjids.add(row[i])

print(list(usubjids))


['01-708-1336', '01-710-1368', '01-715-1321', '01-709-1237', '01-701-1444', '01-718-1371', '01-710-1083', '01-710-1187', '01-714-1035', '01-710-1021', '01-710-1183', '01-715-1319', '01-711-1283', '01-715-1155', '01-701-1341', '01-710-1060', '01-708-1213', '01-716-1364', '01-708-1316', '01-710-1358', '01-703-1086', '01-708-1348', '01-710-1354', '01-716-1298', '01-711-1012', '01-711-1036', '01-705-1058', '01-705-1282', '01-711-1284', '01-703-1439', '01-710-1376', '01-710-1154', '01-717-1174', '01-710-1270', '01-701-1275', '01-709-1088', '01-701-1146', '01-714-1425', '01-701-1148', '01-701-1440', '01-710-1249', '01-711-1143', '01-708-1032', '01-713-1256', '01-716-1024', '01-718-1172', '01-705-1292', '01-701-1234', '01-701-1345', '01-707-1037', '01-708-1087', '01-704-1388', '01-715-1405', '01-701-1287', '01-705-1199', '01-718-1250', '01-701-1153', '01-716-1418', '01-710-1314', '01-717-1109', '01-718-1150', '01-701-1356', '01-709-1238', '01-716-1373', '01-710-1070', '01-701-1181', '01-701-1

# データの書き換え

In [8]:
Target_data = [
["DM", "01-703-1096",   0, "AGE", 49],
["LB", "01-703-1042",   3, "LBORRES", "135"],
["LB", "01-703-1042",   4, "LBORRES", "145"],
["LB", "01-703-1086",  37, "LBORRES", "1"],
["LB", "01-703-1086",  72, "LBORRES", "1.2"],
["LB", "01-703-1086", 102, "LBORRES", "1.1"],
["LB", "01-703-1086", 132, "LBORRES", "1"],
["LB", "01-703-1086", 162, "LBORRES", "1.3"],
["LB", "01-703-1086", 197, "LBORRES", "0.9"],
["LB", "01-703-1086", 232, "LBORRES", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESC", "135"],
["LB", "01-703-1042",   4, "LBSTRESC", "145"],
["LB", "01-703-1086",  37, "LBSTRESC", "1"],
["LB", "01-703-1086",  72, "LBSTRESC", "1.2"],
["LB", "01-703-1086", 102, "LBSTRESC", "1.1"],
["LB", "01-703-1086", 132, "LBSTRESC", "1"],
["LB", "01-703-1086", 162, "LBSTRESC", "1.3"],
["LB", "01-703-1086", 197, "LBSTRESC", "0.9"],
["LB", "01-703-1086", 232, "LBSTRESC", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESN", 135],
["LB", "01-703-1042",   4, "LBSTRESN", 145],
["LB", "01-703-1086",  37, "LBSTRESN", 1],
["LB", "01-703-1086",  72, "LBSTRESN", 1.2],
["LB", "01-703-1086", 102, "LBSTRESN", 1.1],
["LB", "01-703-1086", 132, "LBSTRESN", 1],
["LB", "01-703-1086", 162, "LBSTRESN", 1.3],
["LB", "01-703-1086", 197, "LBSTRESN", 0.9],
["LB", "01-703-1086", 232, "LBSTRESN", 0.8],
["LB", "01-703-1042",   3, "LBNRIND", "HIGH"],
["LB", "01-703-1042",   4, "LBNRIND", "HIGH"],
["LB", "01-703-1086",  37, "LBNRIND", "LOW"],
["LB", "01-703-1086",  72, "LBNRIND", "LOW"],
["LB", "01-703-1086", 102, "LBNRIND", "LOW"],
["LB", "01-703-1086", 132, "LBNRIND", "LOW"],
["LB", "01-703-1086", 162, "LBNRIND", "LOW"],
["LB", "01-703-1086", 197, "LBNRIND", "LOW"],
["LB", "01-703-1086", 232, "LBNRIND", "LOW"],
["MH", "01-701-1097",   1, "MHTERM", "Loss of consciousness (Passed out)"],
["MH", "01-701-1097",   1, "MHSTDTC", "2023-01-01"],
["MH", "01-701-1111",   1, "MHTERM", "HEARING LOSS"],
["MH", "01-701-1180",   1, "MHTERM", "DEPRESSION (ANXIETY)"],
["MH", "01-702-1082",   1, "MHTERM", "Premenstrual pain"],
["MH", "01-703-1076",   1, "MHTERM", "Atrioventricular block (scheduled cardiac pacemaker insertion)"],
["MH", "01-703-1279",   1, "MHTERM", "schizophreniform disorders"],
["MH", "01-703-1299",   1, "MHTERM", "Cyclothymic disorder"],
["VS", "01-701-1047",  17, "VSORRES", "121"],
["VS", "01-701-1047",  18, "VSORRES", "124"],
["VS", "01-701-1047",  66, "VSORRES", "185"],
["VS", "01-701-1047",  67, "VSORRES", "183"],
["VS", "01-701-1383",  37, "VSORRES", "98"],
["VS", "01-701-1383", 122, "VSORRES", "160"],
["VS", "01-701-1387",   1, "VSORRES", "146"],
["VS", "01-701-1387",  32, "VSORRES", "72"],
["VS", "01-701-1047",  17, "VSSTRESC", "121"],
["VS", "01-701-1047",  18, "VSSTRESC", "124"],
["VS", "01-701-1047",  66, "VSSTRESC", "185"],
["VS", "01-701-1047",  67, "VSSTRESC", "183"],
["VS", "01-701-1383",  37, "VSSTRESC", "98"],
["VS", "01-701-1383", 122, "VSSTRESC", "160"],
["VS", "01-701-1387",   1, "VSSTRESC", "146"],
["VS", "01-701-1387",  32, "VSSTRESC", "72"],
["VS", "01-701-1047",  17, "VSSTRESN", 121],
["VS", "01-701-1047",  18, "VSSTRESN", 124],
["VS", "01-701-1047",  66, "VSSTRESN", 185],
["VS", "01-701-1047",  67, "VSSTRESN", 183],
["VS", "01-701-1383",  37, "VSSTRESN", 98],
["VS", "01-701-1383", 122, "VSSTRESN", 160],
["VS", "01-701-1387",   1, "VSSTRESN", 146],
["VS", "01-701-1387",  32, "VSSTRESN", 72],
["EX", "01-701-1148",   2, "EXDOSE", 82],
["EX", "01-701-1148",   3, "EXDOSE", 216],
["EX", "01-703-1258",   2, "EXDOSE", 27],
["CM", "01-701-1146",  29, "CMTRT", "PAROXETINE"],
["QS", "01-701-1023",1010, "QSORRES", "PRESENT"],
["QS", "01-701-1023",1012, "QSORRES", "PRESENT"],
["QS", "01-701-1111",5004, "QSORRES", "4"],
["QS", "01-701-1111",5019, "QSORRES", "4"],
["QS", "01-701-1111",5012, "QSORRES", "4"],
["QS", "01-701-1111",5027, "QSORRES", "4"],
["QS", "01-701-1118",6002, "QSORRES", "MARKED IMPROVEMENT"],
["QS", "01-701-1118",6003, "QSORRES", "MARKED WORSENING"],
["QS", "01-701-1181",4018, "QSORRES", "Y"],
["QS", "01-701-1181",4058, "QSORRES", "Y"],
["QS", "01-701-1181",4019, "QSORRES", "Y"],
["QS", "01-701-1181",4059, "QSORRES", "Y"],
["QS", "01-701-1181",4020, "QSORRES", "Y"],
["QS", "01-701-1023",1010, "QSSTRESC", "2"],
["QS", "01-701-1023",1012, "QSSTRESC", "2"],
["QS", "01-701-1111",5004, "QSSTRESC", "4"],
["QS", "01-701-1111",5019, "QSSTRESC", "4"],
["QS", "01-701-1111",5012, "QSSTRESC", "4"],
["QS", "01-701-1111",5027, "QSSTRESC", "4"],
["QS", "01-701-1118",6002, "QSSTRESC", "1"],
["QS", "01-701-1118",6003, "QSSTRESC", "7"],
["QS", "01-701-1181",4018, "QSSTRESC", "1"],
["QS", "01-701-1181",4058, "QSSTRESC", "1"],
["QS", "01-701-1181",4019, "QSSTRESC", "1"],
["QS", "01-701-1181",4059, "QSSTRESC", "1"],
["QS", "01-701-1181",4020, "QSSTRESC", "1"],
["QS", "01-701-1023",1010, "QSSTRESN", 2],
["QS", "01-701-1023",1012, "QSSTRESN", 2],
["QS", "01-701-1111",5004, "QSSTRESN", 4],
["QS", "01-701-1111",5019, "QSSTRESN", 4],
["QS", "01-701-1111",5012, "QSSTRESN", 4],
["QS", "01-701-1111",5027, "QSSTRESN", 4],
["QS", "01-701-1118",6002, "QSSTRESN", 1],
["QS", "01-701-1118",6003, "QSSTRESN", 7],
["QS", "01-701-1181",4018, "QSSTRESN", 1],
["QS", "01-701-1181",4058, "QSSTRESN", 1],
["QS", "01-701-1181",4019, "QSSTRESN", 1],
["QS", "01-701-1181",4059, "QSSTRESN", 1],
["QS", "01-701-1181",4020, "QSSTRESN", 1],
["QS", "01-701-1118",6001, "QSDTC", "2014-07-08"],
["QS", "01-701-1118",6001, "QSDY", 119],
["AE", "01-701-1015",   3, "AESER", "Y"],
["AE", "01-701-1015",   3, "AESHOSP", "Y"],
["AE", "01-701-1015",   3, "AESTDTC", "2014-01-11"],
["AE", "01-701-1015",   3, "AEENDTC", "2014-01-09"],
["AE", "01-701-1015",   3, "AESTDY", 10],
["AE", "01-701-1015",   3, "AEENDY", 8],
["AE", "01-701-1028",   1, "AETERM", "PARKINSON'S DISEASE"],
["AE", "01-701-1028",   1, "AESTDTC", "2013-07-01"],
["AE", "01-701-1028",   1, "AESTDY", -17],
["AE", "01-701-1034",   2, "AETERM", "MALIGNANT HYPERTENSION"],
["AE", "01-701-1047",   4, "AETERM", "HYPERTENSION"],
["AE", "01-701-1363",   1, "AESTDTC", "2013-06-15"],
["AE", "01-701-1363",   1, "AEENDTC", "2013-06-14"],
["AE", "01-701-1363",   1, "AESTDY", 17],
["AE", "01-701-1363",   1, "AEENDY", 16],
["AE", "01-701-1047",   3, "AEENDTC", "2013-03-05"],
["AE", "01-701-1047",   3, "AEENDY", 22],
["AE", "01-701-1383",  12, "AETERM", "BLOOD PRESSURE INCREASED"],
["AE", "01-701-1153",   2, "AEACN", "DRUG WITHDRAWN"],
["AE", "01-701-1180",   6, "AETERM", "SUDDEN DEATH"],
["AE", "01-703-1258",   2, "AESEV", "SEVERE"],
["AE", "01-703-1258",   2, "AESTDTC", "2012-08-01"],
["AE", "01-703-1258",   2, "AEENDTC", "2012-10-01"],
["AE", "01-703-1258",   2, "AESTDY", 13],
["AE", "01-703-1258",   2, "AEENDY", 74],
["AE", "01-703-1258",   5, "AESEV", "MODERATE"],
["AE", "01-703-1258",   5, "AESER", "Y"],
["AE", "01-703-1258",   5, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-703-1258",   5, "AESLIFE", "Y"],
["AE", "01-703-1258",   5, "AESTDTC", "2012-10-02"],
["AE", "01-703-1258",   5, "AEENDTC", "2012-12-31"],
["AE", "01-703-1258",   2, "AESTDY", 75],
["AE", "01-703-1258",   2, "AEENDY", 165],
["AE", "01-703-1335",   1, "AETERM", "MULTIPLE SCLEROSIS RELAPSE"],
["AE", "01-703-1335",   1, "AESTDTC", "2014-04-01"],
["AE", "01-703-1335",   1, "AEENDTC", "2014-05-01"],
["AE", "01-703-1335",   1, "AESTDY", 15],
["AE", "01-703-1335",   1, "AEENDY", 46],
["AE", "01-703-1403",   2, "AETERM", "MYASTHENIA GRAVIS AGGRAVATED"],
["AE", "01-704-1008",   1, "AETERM", "TREMOR IN HANDS, LEGS"],
["AE", "01-704-1008",   1, "AEREL", "NONE"],
["AE", "01-704-1008",   1, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   1, "AESTDY", -225],
["AE", "01-704-1008",   3, "AETERM", "MUSCLE STIFFNESS"],
["AE", "01-704-1008",   3, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   3, "AESTDY", -225],
["AE", "01-704-1008",   2, "AETERM", "SLOWNESS of MOVEMENT"],
["AE", "01-704-1008",   2, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   2, "AESTDY", -225],
["AE", "01-704-1009",   6, "AETERM", "CHRONIC KIDNEY DISEASE"],
["AE", "01-704-1009",   6, "AESER", "Y"],
["AE", "01-704-1009",   6, "AESLIFE", "Y"],
["AE", "01-704-1010",   1, "AETERM", "DIABETES MELLITUS"],
["AE", "01-704-1010",   1, "AESER", "Y"],
["AE", "01-704-1010",   1, "AESLIFE", "Y"],
["AE", "01-704-1017",   4, "AETERM", "LATE EFFECTS OF CEREBRAL INFRACTION"],
["AE", "01-704-1017",   4, "AESEV", "SEVERE",],
["AE", "01-704-1017",   4, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   4, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   4, "AESTDY", 14],
["AE", "01-704-1017",   4, "AEENDY", 44],
["AE", "01-704-1017",   3, "AETERM", "BRAIN DEATH"],
["AE", "01-704-1017",   3, "AESEV", "SEVERE",],
["AE", "01-704-1017",   3, "AESTDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AESTDY", 44],
["AE", "01-704-1017",   3, "AEENDY", 44],
["AE", "01-704-1017",   1, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-704-1017",   1, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   1, "AEENDTC", "2013-11-19"],
["AE", "01-704-1017",   1, "AESTDY", 14],
["AE", "01-704-1017",   1, "AEENDY", 45],
["AE", "01-704-1017",   1, "AEACN", "DRUG WITHDRAWN"]
]

dataset_list_updated = dataset_list

for l in Target_data:
  #print(l)
  dataset_list_updated = data_update(dataset_list_updated, l[0], l[1], l[2], l[3], l[4])

USUBJID '01-703-1096' の 'AGE' を '49' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBORRES' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBORRES' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBORRES' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBORRES' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '162' の 'LBORRES' を '1.3' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '197' の 'LBORRES' を '0.9' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '232' の 'LBORRES' を '0.8' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBSTRESC' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBSTRESC' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBSTRESC' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBSTRESC' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBSTRESC' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBSTRESC' を '1' に更

In [9]:
# 更新データ確認
compare_data(dataset_list, dataset_list_updated)

with open("dataset_list_updated.json", "w") as f:
  json.dump(dataset_list_updated, f)

--- ItemGroupOID: AE ---
USUBJID = 01-701-1015, AESEQ = 3:
  Updated:
    AEENDTC: '2014-01-11' -> '2014-01-09'
    AEENDY: 10 -> 8
    AESER: 'N' -> 'Y'
    AESHOSP: 'N' -> 'Y'
    AESTDTC: '2014-01-09' -> '2014-01-11'
    AESTDY: 8 -> 10

USUBJID = 01-701-1028, AESEQ = 1:
  Updated:
    AESTDTC: '2013-07-21' -> '2013-07-01'
    AESTDY: 3 -> -17
    AETERM: 'APPLICATION SITE ERYTHEMA' -> "PARKINSON'S DISEASE"

USUBJID = 01-701-1034, AESEQ = 2:
  Updated:
    AETERM: 'FATIGUE' -> 'MALIGNANT HYPERTENSION'

USUBJID = 01-701-1047, AESEQ = 3:
  Updated:
    AEENDTC: '' -> '2013-03-05'
    AEENDY: None -> 22

USUBJID = 01-701-1047, AESEQ = 4:
  Updated:
    AETERM: 'BUNDLE BRANCH BLOCK LEFT' -> 'HYPERTENSION'

USUBJID = 01-701-1153, AESEQ = 2:
  Updated:
    AEACN: '' -> 'DRUG WITHDRAWN'

USUBJID = 01-701-1180, AESEQ = 6:
  Updated:
    AETERM: 'MICTURITION URGENCY' -> 'SUDDEN DEATH'

USUBJID = 01-701-1363, AESEQ = 1:
  Updated:
    AEENDTC: '2013-06-15' -> '2013-06-14'
    AEENDY: 17 -> 16

# LLMへの送信

In [10]:
!pip install sseclient-py
import requests
import sseclient
from IPython.display import display, Markdown

from google.colab import userdata
api_key = userdata.get('Dify_DatasetJSON')
user_id = 'JPMA_Sample'

## 関数定義

In [11]:
import json
import time
import requests
import sseclient

# 定数
DIFY_API_URL = 'https://api.dify.ai/v1/workflows/run'
CONTENT_TYPE_JSON = 'application/json'

def call_dify_api(api_key: str, payload: dict, stream: bool = False) -> requests.Response:
    """Dify APIを呼び出す共通関数"""
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': CONTENT_TYPE_JSON
    }
    try:
        response = requests.post(DIFY_API_URL, headers=headers, json=payload, stream=stream)
        response.raise_for_status()  # HTTPエラーが発生した場合に例外を発生させる
        return response
    except requests.exceptions.RequestException as e:
        print(f"API呼び出しエラー: {e}")
        raise

def run_dify_workflow(api_key: str, workflow_inputs: dict, user_id: str, streaming: bool = False) -> dict | sseclient.Event:
    """Difyワークフローを実行する

    Args:
        api_key: Dify APIキー
        workflow_inputs: ワークフローへの入力
        user_id: ユーザーID
        streaming: ストリーミングモードで実行するかどうか (Falseの場合はブロッキングモード)

    Returns:
        ストリーミングモードの場合はsseclient.Eventのイテレータ、
        ブロッキングモードの場合はAPIのレスポンスのJSONを辞書型で返す
    """
    payload = {
        'inputs': workflow_inputs,
        'response_mode': 'streaming' if streaming else 'blocking',
        'user': user_id
    }
    response = call_dify_api(api_key, payload, stream=streaming)
    if streaming:
        client = sseclient.SSEClient(response)
        return client.events()
    else:
        return response.json()

def safe_print_event_data(event: sseclient.Event):
    """
    与えられたSSEイベントデータから、存在する場合に特定の値を出力します。
    キーが存在しない場合は何も出力しません。Statusが存在する場合にのみTitleと結合させて表示します。

    Args:
        event: イベントデータを含むSSEイベントオブジェクト。event.data属性がJSON文字列であることを想定。
    """
    try:
        data = json.loads(event.data)

        if 'event' in data:
            print(f"Event: {data['event']}")

        if 'data' in data:
            title = data['data'].get('title')
            status = data['data'].get('status')
            if title is not None and status is not None:
                print(f"Node: {title} ({status})")
            elif title is not None:
                print(f"Node: {title}") # Statusが存在しない場合はTitleのみ表示

            if 'error' in data['data']:
                print(f"Error: {data['data']['error']}")
            if 'elapsed_time' in data['data']:
                print(f"Elapsed time: {data['data'].get('elapsed_time')}")
            if 'total_tokens' in data['data']:
                print(f"Total tokens: {data['data'].get('total_tokens')}")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON event data: {e}")
    except AttributeError as e:
        print(f"Error accessing event data attribute: {e}")

def run_workflow_with_retry(api_key: str, workflow_inputs: dict, user_id: str, max_retries: int = 3, retry_delay: int = 20):
    """ワークフローを実行し、エラー発生時にリトライを行う (ストリーミングモード専用)"""
    for retry in range(max_retries + 1):
        print(f"--- 試行回数: {retry + 1} ---")
        success = True
        try:
            for event in run_dify_workflow(api_key, workflow_inputs, user_id, streaming=True):
                safe_print_event_data(event)
                try:
                    event_data = json.loads(event.data)
                    if event_data.get('data', {}).get('error') is not None:
                        print(f"エラーが検出されました: {event_data['data']['error']}")
                        success = False
                        break
                except json.JSONDecodeError:
                    print("JSONデコードエラーが発生しました。")
                    success = False
                    break
                print('------')

            if success:
                print("ワークフローが正常に完了しました。")
                return json.loads(event.data)
            elif retry < max_retries:
                print(f"エラーが発生したため、{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

        except requests.exceptions.RequestException as e:
            print(f"APIリクエスト中にエラーが発生しました: {e}")
            success = False
            if retry < max_retries:
                print(f"{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

## プロンプトの作成

In [12]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()


SysPrompt = '''
あなたは、臨床試験データのレビューを支援するAIアシスタントです。以下の前提知識を理解した上で、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援してください。各タスクでは、ユーザープロンプトで指定された役割になりきって回答してください。

**前提知識:**

*   臨床試験においては患者の安全性が最優先され、有害事象の評価は特に重要です。
*   SDTM (Study Data Tabulation Model) は、CDISCによって策定された臨床試験データの標準モデルです。
*   Define.xmlはSDTMデータの構造を記述したメタデータファイルであり、参考情報として使用します。JSONデータ自体の内容、医学的妥当性、プロトコルとの整合性を優先してレビューしてください。
*   SDTMデータは、DM、AE、VS、LBなど、複数のドメイン（データセット）に分かれています。
*   報告されるJSONデータには、データ入力時の間違いが含まれる可能性があります。
*   提供された情報のみに基づいて回答を作成してください。想像やハルシネーションに基づいた回答は作成してはいけません。

**出力形式:**

*   すべての出力はMarkdown形式で作成してください。
*   見出し記号(#)は使用しないでください。

**その他:**

*   JSONデータまたはDefine.xmlの形式が不正な場合は、その旨をエラーメッセージとして出力してください。
'''




UserInput_Task1 = '''
あなたは臨床試験の専門医です。以下の指示に従い、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データのレビューとクエリ作成（必要な場合）を行ってください。

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリーを作成してください。
    *   特に、**異常所見**を中心に簡潔な文章で記載してください。正常範囲内の変動は省略して構いません。
    *   各イベントの日時は、Define.xmlに定義された日付変数などを参考に、正確に特定してください。
*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

    *   **患者ID:**
        *   YYYY年MM月DD日 (Day XX): [有害事象、検査値、バイタルサインなどのイベントを、異常所見を中心に簡潔な文章で記載]

---

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下のJSONデータのレビュー観点に基づき、JSONデータを改めて点検してください。
    *   医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *  **疑義事項がない場合は、クエリを作成する必要はありません。**「疑義事項なし」と回答してください。

*   **JSONデータのレビュー観点 (これらに限定されない):**
    *   **安全性:** 有害事象(AEドメイン)の報告内容は、医学的に妥当であるか？
    *   **医学的妥当性:** 検査値(LBドメイン)の変動、バイタルサイン(VSドメイン)の変動、併用薬(CMドメイン)との相互作用など、時間経過とともに医学的に問題となる点は見られるか？
    *   **有効性:** 特定された主要評価項目および副次評価項目について、その時間的変化は期待される効果と一致しているか？
    *   **その他:** 患者背景(DMドメイン)、既往歴(MHドメイン)、有害事象(AEドメイン)、治療歴(EXドメイン, CMドメイン)などを総合的に考慮し、時間経過を加味して安全性に懸念を生じる事項があれば記載してください。
    *   **プロトコル逸脱 (疑い):** 選択/除外基準、投与量、併用禁止薬、評価スケジュール、有害事象報告などについて、プロトコルからの逸脱の疑いがないか確認してください。（関連ドメイン: DM, MH, EX, CM, LB, VS, AEなど）

*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。クエリがない場合は、「クエリなし」と出力してください。

    *   **患者ID:**
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
            *   **変数名と値:**
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
'''



UserInput_Task2 = '''
あなたはクリニカルデータマネージャーです。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、データ整合性レビューとクエリ作成（必要な場合）を行ってください。

**1. データ整合性レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、データの不整合が疑われる問題点を検出してください。
    *   **特に、以下の点に焦点を当ててレビューしてください。**
        *   **クロスドメイン整合性:** 異なるSDTMドメイン間で、データに矛盾がないか、ドメイン間の関連性が正しく表現されているか。
            *   **具体的な確認例 (これらに限定されない):**
                *   DM.SEXとAEにおける妊娠関連の有害事象
                *   AEの有害事象発現日や治験薬との関連性と、EXの治験薬の投与期間
                *   LBの検査値異常とAEの関連有害事象
                *   VSのバイタルサイン異常とAEの関連有害事象
                *   CM.CMTRTとAE/MHで報告されている疾患・既往歴との矛盾
        *   **単一ドメイン内の整合性:** Define.xmlの定義に照らして、矛盾なく解釈できるデータになっているか、プロトコルに照らしてデータの関連性が正しく表現されているか。
        *   **異常値:** Define.xmlで定義された範囲外、または医学的にありえない値がないか。
        *   **欠損値:** 欠損値の有無と理由（推測できる場合）。多い場合は原因を推測。
        *   **プロトコル逸脱 (データ品質の観点から):** データ入力/収集で、プロトコルからの逸脱（例：必須項目の未入力、不適切な時期のデータ収集）がないか。
    *   Define.xmlとデータの間に不整合がある場合は、「Define.xmlの修正候補」として報告してください。
*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

    *   **全体的なデータ品質の評価:**
        *   総合評価: [例：良好、一部問題あり、要修正]
        *   データクリーニング/再調査が必要な項目: [該当項目を列挙]

    *   **問題点:**（問題がある場合）
        *   **問題No.:**
            *   **変数名と値:**
            *   **矛盾の内容:** [具体的な矛盾の内容を記述]
        *   **問題点の原因（推測）:**
        *   **対応策（提案）:**

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   データ整合性レビューの結果、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **疑義事項がない場合は、クエリを作成する必要はありません。**
*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。クエリがない場合は、「クエリなし」と出力してください。

    *   **患者ID:**
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
            *   **変数名と値:**
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
'''


UserInput_Task3 = '''
あなたは、臨床試験の専門医、データマネージャー、CRAの視点を持つ、プロトコル遵守状況の確認者です。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、プロトコル逸脱の検出とクエリ作成（必要な場合）を行ってください。

**1. プロトコル逸脱の検出:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、プロトコルからの逸脱を検出してください。
    *   Define.xmlは参考情報として活用し、データとプロトコルの内容を比較して逸脱を判断してください。
    *   **検出対象とすべき主要なプロトコル逸脱の例 (これらに限定されない):**
        *   **選択/除外基準違反:** (関連SDTMドメイン: DM, MH など)
        *   **投与量違反:** (関連SDTMドメイン: EX)
        *   **併用禁止薬の使用:** (関連SDTMドメイン: CM)
        *   **評価スケジュール違反:** (関連SDTMドメイン: LB, VS, その他)
        *   **有害事象報告違反**: (関連SDTMドメイン: AE)
*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

    *   **患者ID:**
        *   **逸脱No.:**
            *   **臨床試験結果への影響度合い:**（Critical/Major/Minor）
            *   **変数名と値:**
            *   **逸脱内容:** [具体的な逸脱内容を簡潔に記述。例：被験者XXXは、プロトコルで規定された投与量を超える量の治験薬を投与された]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **判断理由:**

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   プロトコル逸脱を判定するために、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、プロトコル逸脱が臨床試験の評価項目に与える影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **プロトコル逸脱に関する疑義事項がない場合は、クエリを作成する必要はありません。**
*   **出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。クエリがない場合は、「クエリなし」と出力してください。

    *   **患者ID:**
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
            *   **変数名と値:**
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
'''


UserInput_end1 = '''\n---\n\n**データ:**\n\n*   臨床試験データ（JSON形式、SDTM準拠）:\n\n```json\n'''
UserInput_end2 = '''\n```\n\n*   データ定義ファイル（Define.xml）:\n\n```xml\n''' + define_xml + '''```\n'''

In [13]:
def create_workflow_input(ModelName, SysPrompt, UserInput_Task, datasetjson, UserInput_end1, UserInput_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_Task + UserInput_end1 + datasetjson + UserInput_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

## 実行

In [14]:
# ModelNameの設定
#ModelName = 'gemini-2.0-flash'
#ModelName = 'gemini-2.0-flash-exp'
#ModelName = 'gemini-2.0-flash-exp-multi'
ModelName = 'gemini-2.0-pro-exp-02-05'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21-multi'
#ModelName = 'gemini-2.0-flash-thinking-exp'
#ModelName = 'gemini-2.0-flash-thinking-exp-multi'


# データ更新症例の抽出
updated_subjects = []
for l in Target_data:
  updated_subjects.append(l[1])

updated_subjects = list(set(updated_subjects))
print(updated_subjects)


['01-703-1335', '01-701-1023', '01-703-1258', '01-703-1076', '01-703-1096', '01-704-1017', '01-702-1082', '01-701-1387', '01-703-1086', '01-703-1279', '01-701-1028', '01-701-1146', '01-704-1009', '01-701-1148', '01-701-1097', '01-701-1111', '01-701-1047', '01-701-1153', '01-703-1299', '01-701-1181', '01-701-1180', '01-701-1363', '01-704-1010', '01-701-1118', '01-701-1015', '01-701-1034', '01-701-1383', '01-704-1008', '01-703-1403', '01-703-1042']


In [15]:
import pandas as pd

# リトライ付きでワークフローを実行
results_list = []

for subj in updated_subjects:
    datasetjson = filter_data(dataset_list_updated, subj)
    print(f"処理完了：'datasetjson' に USUBJID が {subj} のデータを出力しました。")

    row_data = {'Subject': subj}  # 各行のデータを格納する辞書

    # Task 1 の処理
    workflow_inputs_Task1 = create_workflow_input(ModelName, SysPrompt, UserInput_Task1, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task1 = run_workflow_with_retry(api_key, workflow_inputs_Task1, user_id)
        output_Task1 = result_Task1['data']['outputs']['text']
        display(Markdown(output_Task1))
        row_data['Task1'] = output_Task1
    except Exception as e:
        print(f"Task 1 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task1'] = "Error"

    # Task 2 の処理
    workflow_inputs_Task2 = create_workflow_input(ModelName, SysPrompt, UserInput_Task2, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task2 = run_workflow_with_retry(api_key, workflow_inputs_Task2, user_id)
        output_Task2 = result_Task2['data']['outputs']['text']
        display(Markdown(output_Task2))
        row_data['Task2'] = output_Task2
    except Exception as e:
        print(f"Task 2 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task2'] = "Error"

    # Task 3 の処理
    workflow_inputs_Task3 = create_workflow_input(ModelName, SysPrompt, UserInput_Task3, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
    try:
        result_Task3 = run_workflow_with_retry(api_key, workflow_inputs_Task3, user_id)
        output_Task3 = result_Task3['data']['outputs']['text']
        display(Markdown(output_Task3))
        row_data['Task3'] = output_Task3
    except Exception as e:
        print(f"Task 3 でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task3'] = "Error"

    results_list.append(row_data)

# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

警告：データセット 'CDISCPILOT01.ta' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.te' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ti' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ts' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.tv' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
処理完了：'datasetjson' に USUBJID が 01-703-1335 のデータを出力しました。
--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.31834
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.365342
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.220817
------
Event: node_started
Node: gemini-2.0-pro-exp-02-05 (T: 0.1)
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Ev

**1. 症例サマリー:**

**患者ID: 01-703-1335**

*   2013年12月28日 (Day -79): スクリーニング検査実施。アルブミン、ALP、ALT、AST、ビリルビン、BUN、カルシウム、コレステロール、CK、クレアチニン、尿酸値は正常範囲内。
*   2014年02月21日 (Day -24): 非予定の検査実施。CK (209 U/L) が基準範囲上限を超過。
*   2014年03月15日 (Day -2): スクリーニング検査2実施。血圧、脈拍は正常範囲内。
*   2014年03月17日 (Day 1): ベースライン。ADAS-Cog(11)スコアは57点、NPI-X(9)合計スコアは24点、DAD合計スコアは4%。キサノメリン54mg投与開始。
*   2014年03月30日 (Day 14): Ambulatory ECG 装着。血圧低下（臥位60mmHg、立位60mmHg）、脈拍上昇（臥位88/min、立位96/min）を認める。
*   2014年03月31日 (Day 15): 2週目来院。血圧上昇（臥位88mmHg、立位90mmHg）、脈拍正常化（臥位72/min、立位70/min）を認める。CK (219 U/L) が基準範囲上限を超過。NPI-X(9)合計スコアは10点。同日、多発性硬化症の再発による第2度房室ブロックの有害事象発現。
*   2014年04月01日 (Day 16): キサノメリン81mgへ増量。
*   2014年04月13日 (Day 28): 4週目来院。血圧低下（臥位80mmHg、立位70mmHg）、脈拍上昇（臥位80/min、立位90/min）を認める。CK (134 U/L) は正常範囲内に回復。NPI-X(9)合計スコアは24点。
*   2014年04月15日 (Day 30): Ambulatory ECG 取り外し。血圧、脈拍は正常範囲内。
*   2014年05月01日 (Day 46): 第2度房室ブロックの有害事象が回復/寛解。
*   2014年05月07日 (Day 52): キサノメリン81mg投与終了。
*   2014年05月24日 (Day 69): 6週目来院。血圧、脈拍は正常範囲内。ADAS-Cog(11)スコアは61.5点、NPI-X(9)合計スコアは10点、CIBIC+スコアは5点（最小悪化）。プロトコル違反のため中止。

**2. クエリ:**

**患者ID: 01-703-1335**

*   **クエリNo.: 1**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** AE.AETERM = "MULTIPLE SCLEROSIS RELAPSE", AE.AEDECOD = "ATRIOVENTRICULAR BLOCK SECOND DEGREE"
    *   **医療機関への問い合わせ文面:** 報告された有害事象は「多発性硬化症の再発」と「第2度房室ブロック」ですが、多発性硬化症は除外基準に該当します。本症例の診断と、除外基準12「重篤な神経疾患の診断」への該当について、詳細な情報を確認させてください。
    *   **判断理由:** 患者背景の確認は、安全性評価の基本であり、除外基準抵触の可能性は、臨床試験の信頼性を損なう可能性があるため。
*   **クエリNo.: 2**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** DS.DSDECOD = "PROTOCOL VIOLATION"
    *   **医療機関への問い合わせ文面:** プロトコル違反の内容について詳細な情報を確認させてください。
    *   **判断理由:** プロトコル違反の内容によっては、症例の評価除外が必要となる可能性があるため。
*   **クエリNo.: 3**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** LB.LBTESTCD = "CK", LB.LBORRES = "209", LB.LBDY = -24, LB.VISIT = "UNSCHEDULED 1.1"
    *   **医療機関への問い合わせ文面:** 2014年02月21日(Day -24)の非予定検査でCK値が基準範囲上限を超過していますが、この検査実施理由と臨床的意義について確認させてください。
    *   **判断理由:** CK値の上昇は、有害事象や基礎疾患との関連が考えられるため。
*   **クエリNo.: 4**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** AE.AESTDTC = "2014-04-01", AE.AEENDTC = "2014-05-01", AE.AETERM = "MULTIPLE SCLEROSIS RELAPSE"
    *   **医療機関への問い合わせ文面:** 有害事象「多発性硬化症の再発」の開始日(2014-04-01)と終了日(2014-05-01)が、AEDECOD="ATRIOVENTRICULAR BLOCK SECOND DEGREE"の有害事象の開始日(2014-03-15)と終了日(2014-03-30)と矛盾しているように見えます。どちらの有害事象が「MULTIPLE SCLEROSIS RELAPSE」に該当するか、確認させてください。
    *   **判断理由:** 有害事象の正確な記録は、安全性評価に必須であるため。
*   **クエリNo.: 5**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** AE.AESTDTC = "2014-03-15", AE.AEENDTC = "2014-03-30", AE.AETERM = "ATRIOVENTRICULAR BLOCK SECOND DEGREE"
    *   **医療機関への問い合わせ文面:** 有害事象「第2度房室ブロック」の開始日(2014-03-15)が、SV.SVSTDTC="2014-03-30"のAmbulatory ECG装着日より前になっています。正確な開始日を確認させてください。
    *   **判断理由:** 有害事象の正確な記録は、安全性評価に必須であるため。
*   **クエリNo.: 6**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** QS.QSTESTCD = "CIBIC", QS.QSORRES = "MINIMAL WORSENING", QS.QSDTC = "2014-05-24"
    *   **医療機関への問い合わせ文面:** CIBIC+の評価が「最小悪化」となっていますが、その根拠となった具体的な変化について、詳細な情報を確認させてください。
    *   **判断理由:** CIBIC+は主要評価項目の一つであり、評価の根拠を確認することは重要であるため。
*   **クエリNo.: 7**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** QS.QSTESTCD = "ACTOT", QS.QSSTRESN = 61.5, QS.QSDTC = "2014-05-24"
    *   **医療機関への問い合わせ文面:** ADAS-Cog(11)の合計スコアが、ベースライン(57点)から悪化(61.5点)していますが、その内訳と、悪化の要因について、詳細な情報を確認させてください。
    *   **判断理由:** ADAS-Cog(11)は主要評価項目の一つであり、スコア悪化の要因を確認することは重要であるため。
*   **クエリNo.: 8**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** QS.QSTESTCD = "NPTOT", QS.QSSTRESN = 10 (2014-03-31), 24 (2014-04-13), 10 (2014-05-24)
    *   **医療機関への問い合わせ文面:** NPI-X(9)合計スコアが、ベースライン(24点)から変動していますが、その内訳と、変動の要因について、詳細な情報を確認させてください。
    *   **判断理由:** NPI-X(9)は副次評価項目の一つであり、スコア変動の要因を確認することは重要であるため。
*   **クエリNo.: 9**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** QS.QSTESTCD = "DAITM40", QS.QSORRES = "Y" (2014-03-17), "N" (2014-05-24)
    *   **医療機関への問い合わせ文面:** DADの「一人で安全に家にいられる」の評価が、ベースライン(「はい」)から最終評価(「いいえ」)へ変化していますが、その理由について詳細な情報を確認させてください。
    *   **判断理由:** DADは副次評価項目の一つであり、評価変化の理由を確認することは重要であるため。


--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.04544
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.195309
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.042645
------
Event: node_started
Node: gemini-2.0-pro-exp-02-05 (T: 0.1)
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Eve

**全体的なデータ品質の評価:**

*   総合評価: 一部問題あり
*   データクリーニング/再調査が必要な項目: AE, LB, QS, SUPPAE, SUPPLB

**問題点:**

*   **問題No.1:**
    *   **変数名と値:** AE.AESTDTC = 2014-04-01, AE.AEENDTC = 2014-05-01, AE.AETERM = "MULTIPLE SCLEROSIS RELAPSE"
    *   **矛盾の内容:** AE.AESTDTC (2014-04-01) が AE.AEENDTC (2014-05-01) より後になっている。また、有害事象名が "MULTIPLE SCLEROSIS RELAPSE" であるが、プロトコルでは多発性硬化症の既往は除外基準(EXCL12)となっている。
    *   **問題点の原因（推測）:** AE.AESTDTC, AE.AEENDTCの日付入力ミス、または、患者選択基準の誤り。
    *   **対応策（提案）:** AE.AESTDTC, AE.AEENDTC の日付を再確認する。患者が除外基準に抵触していないか確認する。

*   **問題No.2:**
    *   **変数名と値:** AE.AESTDTC = 2014-03-15, AE.AEENDTC = 2014-03-30, AE.AETERM = "ATRIOVENTRICULAR BLOCK SECOND DEGREE"
    *   **矛盾の内容:** AE.AESTDTC (2014-03-15) が AE.AEDTC (2014-03-31) より前になっている。また、有害事象名が "ATRIOVENTRICULAR BLOCK SECOND DEGREE" であるが、プロトコルでは第二度房室ブロックの既往は除外基準(EXCL16)となっている。
    *   **問題点の原因（推測）:** AE.AESTDTC, AE.AEDTCの日付入力ミス、または、患者選択基準の誤り。
    *   **対応策（提案）:** AE.AESTDTC, AE.AEDTC の日付を再確認する。患者が除外基準に抵触していないか確認する。

*   **問題No.3:**
    *   **変数名と値:** LB.LBTESTCD = "CK", LB.VISITNUM = 1.1, LB.LBNRIND = "HIGH"
    *   **矛盾の内容:** 非定例外来(UNSCHEDULED 1.1)でCK値が基準範囲を超えている(HIGH)。
    *   **問題点の原因（推測）:** 有害事象の発生、または、その他の医学的な理由による検査値異常。
    *   **対応策（提案）:** 基準範囲を超えた理由を確認する。有害事象として報告すべきか検討する。

*   **問題No.4:**
    *   **変数名と値:** QS.QSTESTCD = "ACTOT", QS.VISITNUM = 7, QS.QSSTRESN = 61.5517241379
    *   **矛盾の内容:** QS.QSSTRESN (ADAS-COG(11) Subscore) が小数点以下6桁まで入力されている。Define.xmlでは、QS.QSSTRESNのデータタイプは整数(integer)と定義されている。
    *   **問題点の原因（推測）:** データ入力/転記ミス、または、Define.xmlの定義の誤り。
    *   **対応策（提案）:** Define.xmlの修正候補: QS.QSSTRESNのデータタイプをfloatに変更し、有効桁数を定義する。

*   **問題No.5:**
    *   **変数名と値:** SUPPAE.QNAM = "AETRTEM", SUPPAE.QVAL = "N"
    *   **矛盾の内容:** SUPPAE.QVAL = "N" (治験薬との関連性なし) となっているが、AE.AEREL = "NONE" (因果関係なし) となっている。
    *   **問題点の原因（推測）:** SUPPAE.QVAL または AE.AEREL の入力/評価ミス。
    *   **対応策（提案）:** AETRTEM, AERELの定義と、報告された有害事象と治験薬との関連性を再確認し、矛盾がないように修正する。

*   **問題No.6:**
    *   **変数名と値:** SUPPLB.QNAM = "LBTMSHI", SUPPLB.QVAL
    *   **矛盾の内容:** SUPPLB.QVALの値が、Define.xmlで定義されている表示形式(12.1)と一致しないものがある。
    *   **問題点の原因（推測）:** SUPPLB.QVALの計算/転記ミス、または、Define.xmlの定義の誤り。
    *   **対応策（提案）:** Define.xmlの修正候補: SUPPLB.QVALの表示形式が正しいか確認し、必要に応じて修正する。

**クエリ:**

*   **患者ID:** 01-703-1335
    *   **クエリNo.1:**
        *   **臨床試験結果への影響度合い:** Major
        *   **変数名と値:** AE.AESTDTC = 2014-04-01, AE.AEENDTC = 2014-05-01, AE.AETERM = "MULTIPLE SCLEROSIS RELAPSE"
        *   **医療機関への問い合わせ文面:** AE.AESTDTCが2014-04-01、AE.AEENDTCが2014-05-01と報告されていますが、AESTDTCがAEENDTCよりも後の日付になっています。正しい日付を教えてください。また、有害事象名が "MULTIPLE SCLEROSIS RELAPSE" と報告されていますが、多発性硬化症の既往は除外基準(プロトコル EXCL12)に該当します。患者選択基準を満たしているか、再度ご確認ください。
        *   **判断理由:** 有害事象の発現日と終了日の矛盾、および、除外基準に抵触する可能性のある有害事象の報告は、患者の安全性とデータの信頼性に影響するため。

    *   **クエリNo.2:**
        *   **臨床試験結果への影響度合い:** Major
        *   **変数名と値:** AE.AESTDTC = 2014-03-15, AE.AEENDTC = 2014-03-30, AE.AETERM = "ATRIOVENTRICULAR BLOCK SECOND DEGREE"
        *   **医療機関への問い合わせ文面:** AE.AESTDTCが2014-03-15、AE.AEENDTCが2014-03-30と報告されていますが、AE.AEDTCは2014-03-31と報告されています。正しい日付を教えてください。また、有害事象名が "ATRIOVENTRICULAR BLOCK SECOND DEGREE" と報告されていますが、第二度房室ブロックの既往は除外基準(プロトコル EXCL16)に該当します。患者選択基準を満たしているか、再度ご確認ください。
        *   **判断理由:** 有害事象の発現日と収集日の矛盾、および、除外基準に抵触する可能性のある有害事象の報告は、患者の安全性とデータの信頼性に影響するため。

    *   **クエリNo.3:**
        *   **臨床試験結果への影響度合い:** Minor
        *   **変数名と値:** LB.LBTESTCD = "CK", LB.VISITNUM = 1.1, LB.LBNRIND = "HIGH"
        *   **医療機関への問い合わせ文面:** 非定例外来(Visit 1.1)でCK値が基準範囲を超えていますが、その理由を教えてください。有害事象として報告すべきでしょうか？
        *   **判断理由:** 基準範囲外の値は、有害事象の可能性を示唆するため。

    *   **クエリNo.4:**
        *   **臨床試験結果への影響度合い:** Major
        *   **変数名と値:** SUPPAE.QNAM = "AETRTEM", SUPPAE.QVAL = "N", AE.AEREL = "NONE"
        *   **医療機関への問い合わせ文面:** 有害事象 "MULTIPLE SCLEROSIS RELAPSE" および "ATRIOVENTRICULAR BLOCK SECOND DEGREE" について、SUPPAE.QVAL (AETRTEM) が "N"、AE.AEREL が "NONE" と報告されています。これらの有害事象と治験薬との関連性について、再評価をお願いします。
        *   **判断理由:** 有害事象と治験薬との関連性は、安全性評価の重要な要素であり、矛盾がある場合は確認が必要なため。


--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.039702
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.198966
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.063523
------
Event: node_started
Node: gemini-2.0-pro-exp-02-05 (T: 0.1)
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Event: text_chunk
------
Ev

...

**1. 症例サマリー:**

**患者ID: 01-702-1082**

*   2013年7月3日 (Day -23): スクリーニング検査実施。白血球数 (WBC) 14.77 THOU/uL (高値)、BUN 25 mg/dL (高値)、HCT 49.0% (高値)、ビタミンB12 946 pg/mL (高値)。
*   2013年7月24日 (Day -2): 非定型検査1.1実施。ALT 37 U/L (高値)、AST 44 U/L (高値)、BUN 27 mg/dL (高値)。白血球数増加、尿検査異常の有害事象報告 (いずれも軽度)。
*   2013年7月26日 (Day 1): ベースライン評価。白血球数増加、尿検査異常の有害事象は継続中 (いずれも軽度)。低用量Xanomeline投与開始。
*   2013年8月8日 (Day 14): Week 2評価。白血球数増加、尿検査異常の有害事象は回復/寛解。クレアチンキナーゼ (CK) 320 U/L (高値)。
*   2013年9月6日 (Day 43): Week 6評価。直腸出血の有害事象報告 (軽度)。白血球数 (WBC) 13.18 THOU/uL (高値)。
*   2013年9月9日 (Day 46): 適用部位の発疹の有害事象報告 (軽度)。
*   2013年9月24日 (Day 61): 適用部位の発疹の有害事象は回復/寛解。
*   2013年9月28日 (Day 65): Week 8評価。直腸出血の有害事象は回復/寛解。白血球数 (WBC) 10.76 THOU/uL (高値)、CK 177 U/L (高値)。
*   2013年10月12日 (Day 79): Week 10 (電話)評価。皮膚刺激の有害事象報告 (中等度)。
*   2013年10月31日 (Day 98): 皮膚刺激の有害事象は回復/寛解。
*   2013年11月17日 (Day 115): Week 12評価。試験中止 (被験者による中止)。中止理由は「パッチが不便でかゆい。錠剤を希望」。白血球数 (WBC) 14.38 THOU/uL (高値)、尿中ケトン体 1 (異常)。

**2. クエリ:**

**患者ID: 01-702-1082**

*   **クエリNo.: 1**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** AE.AETERM = "RECTAL HAEMORRHAGE", AE.AESEV = "MILD", AE.AEOUT = "NOT RECOVERED/NOT RESOLVED", AE.AESTDTC = "2013-09-02", AE.AEENDTC = "2013-09-06"
    *   **医療機関への問い合わせ文面:** 2013年9月6日に報告された軽度の直腸出血 (AE.AETERM = "RECTAL HAEMORRHAGE") は、2013年9月6日に回復したと報告されていますが (AESEQ=7)、2013年9月28日の記録では未回復/未寛解とされています (AESEQ=8)。どちらが正しいか、確認してください。
    *   **判断理由:** 同一の有害事象 (直腸出血) に対して、異なる転帰 (AEOUT) が報告されており、矛盾しているため。
*   **クエリNo.: 2**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** AE.AETERM = "WHITE BLOOD CELL COUNT INCREASED", AE.AESEV = "MILD", AE.AEOUT = "NOT RECOVERED/NOT RESOLVED", AE.AESTDTC = "2013-07-07", AE.AEENDTC = "2013-07-26"
    *   **医療機関への問い合わせ文面:** 2013年7月24日に報告された白血球数増加 (AE.AETERM = "WHITE BLOOD CELL COUNT INCREASED") は、2013年7月26日に回復したと報告されていますが (AESEQ=1)、2013年8月8日の記録では未回復/未寛解とされています (AESEQ=5)。どちらが正しいか、確認してください。
    *   **判断理由:** 同一の有害事象 (白血球数増加) に対して、異なる転帰 (AEOUT) が報告されており、矛盾しているため。
*   **クエリNo.: 3**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** AE.AETERM = "URINE ANALYSIS ABNORMAL", AE.AESEV = "MILD", AE.AEOUT = "NOT RECOVERED/NOT RESOLVED", AE.AESTDTC = "2013-07-07", AE.AEENDTC = "2013-07-24"
    *   **医療機関への問い合わせ文面:** 2013年7月24日に報告された尿検査異常 (AE.AETERM = "URINE ANALYSIS ABNORMAL") は、2013年7月24日に回復したと報告されていますが (AESEQ=3)、2013年7月26日の記録では未回復/未寛解とされています (AESEQ=4)。どちらが正しいか、確認してください。
    *   **判断理由:** 同一の有害事象 (尿検査異常) に対して、異なる転帰 (AEOUT) が報告されており、矛盾しているため。
*   **クエリNo.: 4**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "WBC", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** スクリーニング時 (2013-07-03)、Week 6 (2013-09-06)、Week 8 (2013-09-28)、Week 12 (2013-11-17) の白血球数 (WBC) が高値 (LBNRIND = "HIGH") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** 白血球数 (WBC) は安全性評価の重要な指標であり、基準範囲を逸脱する変動は、有害事象の可能性を示唆するため。
*   **クエリNo.: 5**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "CK", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** Week 2 (2013-08-08) および Week 8 (2013-09-28) のクレアチンキナーゼ (CK) が高値 (LBNRIND = "HIGH") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** クレアチンキナーゼ (CK) は安全性評価の重要な指標であり、基準範囲を逸脱する変動は、有害事象 (特に筋障害) の可能性を示唆するため。
*   **クエリNo.: 6**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "BUN", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** スクリーニング時 (2013-07-03)、非定型検査1.1 (2013-07-24)、Week 6 (2013-09-06) の血中尿素窒素 (BUN) が高値 (LBNRIND = "HIGH") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** 血中尿素窒素 (BUN) は安全性評価の重要な指標であり、基準範囲を逸脱する変動は、有害事象 (特に腎機能障害) の可能性を示唆するため。
*   **クエリNo.: 7**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "VITB12", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** スクリーニング時 (2013-07-03) のビタミンB12が基準範囲上限を超過しています (LBNRIND = "HIGH")。医学的に問題となるか、評価してください。
    *   **判断理由:** プロトコルでは、ビタミンB12が基準範囲下限を下回る場合を除外基準としていますが、上限を超える場合の規定がありません。医学的評価が必要です。
*   **クエリNo.: 8**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "PROT", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** Week 12 (2013-11-17) の総蛋白 (PROT) が高値 (LBNRIND = "HIGH") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** 総蛋白 (PROT) は安全性評価の重要な指標であり、基準範囲を逸脱する変動は、有害事象の可能性を示唆するため。
*   **クエリNo.: 9**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** LB.LBTESTCD = "HCT", LB.LBSTRESN, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** スクリーニング時 (2013-07-03) のヘマトクリット (HCT) が高値 (LBNRIND = "HIGH") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** ヘマトクリット (HCT) は安全性評価の重要な指標であり、基準範囲を逸脱する変動は、有害事象の可能性を示唆するため。
*   **クエリNo.: 10**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** LB.LBTESTCD = "KETONES", LB.LBSTRESC, LB.LBNRIND
    *   **医療機関への問い合わせ文面:** Week 12 (2013-11-17) の尿中ケトン体が異常 (LBNRIND = "ABNORMAL") です。医学的に問題となる変動か、評価してください。
    *   **判断理由:** 尿中ケトン体は安全性評価の指標であり、異常値は有害事象の可能性を示唆するため。ただし、他の検査値と比較して、臨床試験結果への影響はMinorと判断しました。
*   **クエリNo.: 11**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** DS.DSTERM, DS.DSDECOD
    *   **医療機関への問い合わせ文面:** 2013年11月17日に「被験者による中止 (WITHDRAWAL BY SUBJECT)」が発生しています。中止理由が「パッチが不便でかゆい。錠剤を希望」とのことですが、プロトコルでは、皮膚刺激は有害事象として報告されるべきです。皮膚刺激は有害事象として報告されていましたか。報告されていれば、その詳細 (発現日、消失日、重症度、処置、転帰など) を教えてください。
    *   **判断理由:** 被験者の中止理由が有害事象 (皮膚刺激) に関連する可能性があり、有害事象の報告漏れの疑いがあるため。
*   **クエリNo.: 12**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** EX.EXSTDTC, EX.EXENDTC
    *   **医療機関への問い合わせ文面:** 2013年8月9日から2013年10月13日までの投与記録 (EX.EXSTDTC, EX.EXENDTC) がありません。投与状況を確認してください。
    *   **判断理由:** 投与記録 (EXドメイン) は、被験者の曝露状況を把握するための重要な情報であり、欠損はデータの解釈に影響を与える可能性があるため。
*   **クエリNo.: 13**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** SV.VISITNUM = 1.1, SV.VISIT = "UNSCHEDULED 1.1", SV.SVSTDTC = "2013-07-24", SV.SVENDTC = "2013-07-24"
    *   **医療機関への問い合わせ文面:** 非定型来院1.1 (2013年7月24日) が発生した理由を教えてください。
    *   **判断理由:** 非定型来院は、有害事象やプロトコル逸脱など、特別な理由で発生する可能性があるため。
*   **クエリNo.: 14**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** SV.VISITNUM = 8.1, SV.VISIT = "WEEK 10 (T)", SV.SVSTDTC = "2013-10-12", SV.SVENDTC = "2013-10-12"
    *   **医療機関への問い合わせ文面:** Week 10 (電話) (2013年10月12日) は、プロトコルで規定された来院ですか。
    *   **判断理由:** Define.xmlとプロトコルで規定された来院スケジュールに、Week 10 (電話) の記載がありません。
*   **クエリNo.: 15**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** MH.MHSTDTC
    *   **医療機関への問い合わせ文面:** MH.MHSTDTC = "" (空欄) となっている、"Premenstrual pain" および "AORTIC STENOSIS" の発現日を特定してください。
    *   **判断理由:** 既往歴の発現日は、有害事象が治験薬投与前から存在していたかどうかを判断する上で重要であるため。


--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.047456
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.202848
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.092143
------
Event: node_started
Node: gemini-2.0-pro-exp-02-05 (T: 0.1)
------
Event: node_finished
Node: gemini-2.0-pro-exp-02-05 (T: 0.1) (failed)
Error: [models] Rate Limit Error, 429 Resource has been exhausted (e.g. check quota).
Elapsed time: 0.442347
エラーが検出されました: [models] Rate Limit Error, 429 Resource has been exhausted (e.g. check quota).
エラーが発生したため、20秒後に再試行します...
--- 試行回数: 2 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.045907
------
Event: node_started
Node: テキスト抽出ツール
--

**1. 症例サマリー:**

**患者ID: 01-701-1387**

*   2014年2月23日 (Day -17): スクリーニング1。87歳女性。既往歴にアルツハイマー病(2003年8月17日発症)、多発性関節炎、鼻漏、肺炎(1940年)、基底細胞癌(2011年3月)、姿勢性めまい、失神感(2011年)、義歯装着、白内障、末梢性浮腫、手のしびれ、抜歯(2003年)。併用薬としてECOTRIN(アスピリン)を2012年から頓用。スクリーニング時の検査で、白血球数、ヘモグロビン、ヘマトクリット、血小板数は正常範囲内。生化学検査では、ナトリウム143 mEq/L (基準範囲 135-145 mEq/L)、カリウム 5.1 mEq/L (基準範囲 3.4-5.4 mEq/L)と軽度高値。Modified Hachinski Ischemic Scaleは0点、MMSEは11点。
*   2014年3月8日 (Day -4): スクリーニング2。バイタルサインは、臥位血圧153/81 mmHg、脈拍72 bpm。
*   2014年3月12日 (Day 1): ベースライン。プラセボ投与開始。ADAS-Cog(11)は44点、NPI-X(9)は26点。DADは全項目で自立度低下(スコア0または1)。
*   2014年3月18日 (Day 7): 下痢、発汗過多の有害事象発現（いずれも軽度）。
*   2014年3月25日 (Day 14): Week 2。プロトコル違反のため試験中止。有害事象は回復/寛解。ADAS-Cog(11)は44点、CIBIC+は3点(軽度改善)、NPI-X(9)は2点。DADは全項目で自立度低下(スコア0または1)。生化学検査でナトリウム146 mEq/L (基準範囲 135-145 mEq/L)と軽度高値。
* 2014年8月27日 (Day 169): 最終観察。ADAS-Cog(11)は59点、CIBIC+は6点(中等度悪化)、NPI-X(9)は16点。DADはDAITM15, DAITM16, DAITM17, DAITM30, DAITM36, DAITM37で自立度改善(スコア1)。バイタルサインは、臥位血圧175/69 mmHg、脈拍70 bpm。

**2. クエリ:**

**患者ID: 01-701-1387**

*   **クエリNo.: 1**
    *   **臨床試験結果への影響度合い:** Major
    *   **変数名と値:** DS.DSDECOD = "PROTOCOL VIOLATION"
    *   **医療機関への問い合わせ文面:** プロトコル違反の内容について、詳細な情報を提供してください。
    *   **判断理由:** 患者がプロトコル違反により試験を中止しているが、具体的な違反内容が不明であるため。プロトコル違反の内容によっては、患者の安全性やデータの解釈に影響を与える可能性がある。
*   **クエリNo.: 2**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** VS.VSORRES (DIABP) = "146" (2014-02-23, SUPINE)
    *   **医療機関への問い合わせ文面:** 2014年2月23日の臥位での拡張期血圧(DIABP)が146mmHgと報告されていますが、これは収縮期血圧(SYSBP)の誤入力ではないでしょうか？確認をお願いします。
    *   **判断理由:** 同日の収縮期血圧(SYSBP)が72mmHgと報告されており、拡張期血圧(DIABP)が146mmHgであることは医学的に矛盾しているため。
*   **クエリNo.: 3**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** CM.CMTRT = "ECOTRIN", CM.CMDECOD = "UNCODED"
    *   **医療機関への問い合わせ文面:** ECOTRINの標準化された医薬品名(CMDECOD)をWHO Drug Dictionaryでコーディングしてください。
    *   **判断理由:** CMドメインのCMTRT(併用薬)が報告されているが、CMDECOD(標準化された医薬品名)が"UNCODED"となっているため。
*   **クエリNo.: 4**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "ALZHEIMER'S DISEASE", MH.MHDECOD = ""
    *   **医療機関への問い合わせ文面:** ALZHEIMER'S DISEASEの標準病名(MHDECOD)をMedDRAでコーディングしてください。
    *   **判断理由:** MHドメインのMHTERM(既往歴)が報告されているが、MHDECOD(標準病名)が空欄になっているため。
*   **クエリNo.: 5**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0261", MH.MHDECOD = "HLT_0134"
    *   **医療機関への問い合わせ文面:** VERBATIM_0261の標準病名(MHDECOD)がHLTコード(HLT_0134)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 6**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0827", MH.MHDECOD = "HLT_0635"
    *   **医療機関への問い合わせ文面:** VERBATIM_0827の標準病名(MHDECOD)がHLTコード(HLT_0635)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 7**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0919", MH.MHDECOD = "HLT_0251"
    *   **医療機関への問い合わせ文面:** VERBATIM_0919の標準病名(MHDECOD)がHLTコード(HLT_0251)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 8**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0951", MH.MHDECOD = "HLT_0332"
    *   **医療機関への問い合わせ文面:** VERBATIM_0951の標準病名(MHDECOD)がHLTコード(HLT_0332)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 9**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0958", MH.MHDECOD = "HLT_0214"
    *   **医療機関への問い合わせ文面:** VERBATIM_0958の標準病名(MHDECOD)がHLTコード(HLT_0214)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 10**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_0982", MH.MHDECOD = "HLT_0619"
    *   **医療機関への問い合わせ文面:** VERBATIM_0982の標準病名(MHDECOD)がHLTコード(HLT_0619)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 11**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1022", MH.MHDECOD = "HLT_0246"
    *   **医療機関への問い合わせ文面:** VERBATIM_1022の標準病名(MHDECOD)がHLTコード(HLT_0246)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 12**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1340", MH.MHDECOD = "HLT_0090"
    *   **医療機関への問い合わせ文面:** VERBATIM_1340の標準病名(MHDECOD)がHLTコード(HLT_0090)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 13**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1392", MH.MHDECOD = "HLT_0661"
    *   **医療機関への問い合わせ文面:** VERBATIM_1392の標準病名(MHDECOD)がHLTコード(HLT_0661)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 14**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1716", MH.MHDECOD = "HLT_0202"
    *   **医療機関への問い合わせ文面:** VERBATIM_1716の標準病名(MHDECOD)がHLTコード(HLT_0202)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 15**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1732", MH.MHDECOD = "HLT_0014"
    *   **医療機関への問い合わせ文面:** VERBATIM_1732の標準病名(MHDECOD)がHLTコード(HLT_0014)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 16**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** MH.MHTERM = "VERBATIM_1738", MH.MHDECOD = "HLT_0209"
    *   **医療機関への問い合わせ文面:** VERBATIM_1738の標準病名(MHDECOD)がHLTコード(HLT_0209)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** MHドメインのMHDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 17**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** AE.AETERM = "DIARRHOEA", AE.AEDECOD = "HLT_0148"
    *   **医療機関への問い合わせ文面:** DIARRHOEAの標準病名(AEDECOD)がHLTコード(HLT_0148)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** AEドメインのAEDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。
*   **クエリNo.: 18**
    *   **臨床試験結果への影響度合い:** Minor
    *   **変数名と値:** AE.AETERM = "HYPERHIDROSIS", AE.AEDECOD = "HLT_0587"
    *   **医療機関への問い合わせ文面:** HYPERHIDROSISの標準病名(AEDECOD)がHLTコード(HLT_0587)になっています。MedDRAのPTコードでコーディングしてください。
    *   **判断理由:** AEドメインのAEDECOD(標準病名)がHLTコードになっているため。MedDRAのPTコードでコーディングする必要がある。


--- 試行回数: 1 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.037948
------
Event: node_started
Node: テキスト抽出ツール
------
Event: node_finished
Node: テキスト抽出ツール (succeeded)
Error: None
Elapsed time: 0.169501
------
Event: node_started
Node: IF/ELSE
------
Event: node_finished
Node: IF/ELSE (succeeded)
Error: None
Elapsed time: 0.059565
------
Event: node_started
Node: gemini-2.0-pro-exp-02-05 (T: 0.1)
------
Event: node_finished
Node: gemini-2.0-pro-exp-02-05 (T: 0.1) (failed)
Error: [models] Rate Limit Error, 429 Resource has been exhausted (e.g. check quota).
Elapsed time: 0.423845
エラーが検出されました: [models] Rate Limit Error, 429 Resource has been exhausted (e.g. check quota).
エラーが発生したため、20秒後に再試行します...
--- 試行回数: 2 ---
Event: workflow_started
------
Event: node_started
Node: 開始
------
Event: node_finished
Node: 開始 (succeeded)
Error: None
Elapsed time: 0.041553
------
Event: node_started
Node: テキスト抽出ツール
--

KeyboardInterrupt: 

In [16]:
# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

,Subject,Task1,Task2,Task3
0,01-703-1335,**1. 症例サマリー:**\n\n**患者ID: 01-703-1335**\n\n* ...,**全体的なデータ品質の評価:**\n\n* 総合評価: 一部問題あり\n* データ...,**1. プロトコル逸脱の検出**\n\n**患者ID: 01-703-1335**\n\n...
1,01-701-1023,**1. 症例サマリー:**\n\n**患者ID: 01-701-1023**\n\n* ...,**Overall Data Quality Evaluation:**\n\n* **...,**1. プロトコル逸脱の検出**\n\n**患者ID: 01-701-1023**\n\n...
2,01-703-1258,**1. 症例サマリー:**\n\n**患者ID: 01-703-1258**\n\n* ...,**Overall Data Quality Evaluation:**\n\n* **...,**Patient ID: 01-703-1258**\n\n* **逸脱No.: 1*...
3,01-703-1076,**1. 症例サマリー:**\n\n* **患者ID: 01-703-1076**\n ...,**全体的なデータ品質の評価:**\n\n* 総合評価: 一部問題あり\n* データ...,**1. プロトコル逸脱の検出**\n\n**患者ID: 01-703-1076**\n\n...
4,01-703-1096,**症例サマリー:**\n\n**患者ID: 01-703-1096**\n\n* 20...,**全体的なデータ品質の評価:**\n\n* 総合評価: 一部問題あり\n* データ...,**1. プロトコル逸脱の検出**\n\n**患者ID: 01-703-1096**\n\n...
5,01-704-1017,**1. 症例サマリー:**\n\n**患者ID: 01-704-1017**\n\n* ...,**全体的なデータ品質の評価:**\n\n* 総合評価: 一部問題あり\n* データ...,以下にプロトコル逸脱の検出結果と、必要に応じて作成したクエリを示します。\n\n**患者ID...
6,01-702-1082,**1. 症例サマリー:**\n\n**患者ID: 01-702-1082**\n\n* ...,Error,Error


In [17]:
def dataframe_to_text(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = row['Task1']
        task2 = row['Task2']
        task3 = row['Task3']

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data
output_text = dataframe_to_text(df_results)

In [18]:
# mdファイルに保存
output_file = 'output_' + ModelName + '.md'  # 保存するファイル名を指定
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(output_text)

print(output_text)

# 01-703-1335
## Task1: Clinical Review Results
**1. 症例サマリー:**

**患者ID: 01-703-1335**

*   2013年12月28日 (Day -79): スクリーニング検査実施。アルブミン、ALP、ALT、AST、ビリルビン、BUN、カルシウム、コレステロール、CK、クレアチニン、尿酸値は正常範囲内。
*   2014年02月21日 (Day -24): 非予定の検査実施。CK (209 U/L) が基準範囲上限を超過。
*   2014年03月15日 (Day -2): スクリーニング検査2実施。血圧、脈拍は正常範囲内。
*   2014年03月17日 (Day 1): ベースライン。ADAS-Cog(11)スコアは57点、NPI-X(9)合計スコアは24点、DAD合計スコアは4%。キサノメリン54mg投与開始。
*   2014年03月30日 (Day 14): Ambulatory ECG 装着。血圧低下（臥位60mmHg、立位60mmHg）、脈拍上昇（臥位88/min、立位96/min）を認める。
*   2014年03月31日 (Day 15): 2週目来院。血圧上昇（臥位88mmHg、立位90mmHg）、脈拍正常化（臥位72/min、立位70/min）を認める。CK (219 U/L) が基準範囲上限を超過。NPI-X(9)合計スコアは10点。同日、多発性硬化症の再発による第2度房室ブロックの有害事象発現。
*   2014年04月01日 (Day 16): キサノメリン81mgへ増量。
*   2014年04月13日 (Day 28): 4週目来院。血圧低下（臥位80mmHg、立位70mmHg）、脈拍上昇（臥位80/min、立位90/min）を認める。CK (134 U/L) は正常範囲内に回復。NPI-X(9)合計スコアは24点。
*   2014年04月15日 (Day 30): Ambulatory ECG 取り外し。血圧、脈拍は正常範囲内。
*   2014年05月01日 (Day 46): 第2度房室ブロックの有害事象が回復/寛解。
*   2014年05月07日 (Day 52): キサノメリン81mg投与終了。
*   2014年05月24日 (Day 69): 6